# Test: `.freeze_jitter()` — jitter alignment across chained resid/reduce functions

`gf_resid()` / `gf_square_resid()` (working copy in `gf_resid_gf_squaresid.R`) and
`gf_reduce()` / `gf_square_reduce()` (`gf_reduce.R`) now pin a fixed seed on the plot's
jitter layer (`.freeze_jitter()`, canonical copy in `freeze_plot.R`) instead of bracketing
the global RNG with `sample()`/`set.seed()`.

- Any number of these functions can be chained, in any order, with any pipe style — all overlays should sit on the jittered dots
- Alignment must survive re-rendering the same plot object
- Diagnosis of the old bug: `test_resid_square_alignment.ipynb`

In [ ]:
#pak::pak("coursekata/coursekata-r")
library(coursekata)
source("../gf_resid_gf_squaresid.R")   # working copy — masks the package versions
source("../gf_reduce.R")

# Small categorical dataset used throughout — matches the original bug report
set.seed(42)
er_small <- sample(er, 12)
m_cat_empty   <- lm(later_anxiety ~ NULL,      data = er_small)
m_cat_complex <- lm(later_anxiety ~ condition, data = er_small)

# Small continuous dataset
set.seed(1)
d <- Fingers[sample(nrow(Fingers), 10), ]
m_empty   <- lm(Thumb ~ NULL,   data = d)
m_complex <- lm(Thumb ~ Height, data = d)

# Confirm the working copies mask library(coursekata)'s versions — both should say R_GlobalEnv
cat("gf_resid:       ", environmentName(environment(gf_resid)), "\n")
cat("gf_square_resid:", environmentName(environment(gf_square_resid)), "\n")

## Single functions on jittered dots

These already worked before the fix and must still work. Run each cell a few times —
the overlays should sit on the dots every time.

In [ ]:
# Test 1: gf_square_resid alone — every square should hang between a dot and the group mean
gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3) %>%
  gf_model(m_cat_complex) %>%
  gf_square_resid(m_cat_complex, color = "blue")

In [ ]:
# Test 2: gf_resid alone — red lines from each dot to its group mean
gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3) %>%
  gf_model(m_cat_complex) %>%
  gf_resid(m_cat_complex, color = "red")

In [ ]:
# Test 3: gf_reduce alone — green lines from grand mean to group means, at each dot's x
gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3) %>%
  gf_model(m_cat_empty) %>%
  gf_model(m_cat_complex) %>%
  gf_reduce(m_cat_complex, color = "forestgreen")

In [ ]:
# Test 4: gf_square_reduce alone
gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3) %>%
  gf_model(m_cat_empty) %>%
  gf_model(m_cat_complex) %>%
  gf_square_reduce(m_cat_complex, color = "forestgreen")

## Previously broken: two or more of these functions chained

Before the fix these chains silently misaligned — squares and lines agreed with each
other but the dots were somewhere else. Run each cell a few times — everything should
sit on the dots every time.

In [ ]:
# Test 5: gf_square_resid then gf_resid — the original bug report
gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3) %>%
  gf_square_resid(m_cat_complex, color = "blue") %>%
  gf_resid(m_cat_complex, color = "red")

In [ ]:
# Test 6: reverse order — gf_resid then gf_square_resid
gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3) %>%
  gf_resid(m_cat_complex, color = "red") %>%
  gf_square_resid(m_cat_complex, color = "blue")

In [ ]:
# Test 7: gf_square_resid then gf_square_reduce — mixing the two source files
gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3) %>%
  gf_square_resid(m_cat_complex, color = "blue") %>%
  gf_square_reduce(m_cat_complex, color = "forestgreen")

In [ ]:
# Test 8: the full chain from the original bug report
gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3) %>%
  gf_square_resid(m_cat_complex, color = "blue") %>%
  gf_model(m_cat_empty) %>%
  gf_model(m_cat_complex) %>%
  gf_square_reduce(m_cat_complex, color = "forestgreen")

In [ ]:
# Test 9: everything at once — full decomposition squares plus both line versions
gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3) %>%
  gf_model(m_cat_empty) %>%
  gf_model(m_cat_complex) %>%
  gf_square_resid(m_cat_empty,    fill = "blue",  color = "blue",  alpha = 0.15) %>%
  gf_square_resid(m_cat_complex,  fill = "red",   color = "red",   alpha = 0.15) %>%
  gf_square_reduce(m_cat_complex, fill = "green", color = "green", alpha = 0.15) %>%
  gf_resid(m_cat_complex, color = "red") %>%
  gf_reduce(m_cat_complex, color = "darkgreen")

## Chaining style shouldn't matter

The old seed bracket behaved differently under lazy pipes vs eager assignments
(see `test_resid_square_alignment.ipynb` section 4). Now all styles give the same result.

In [ ]:
# Test 10: eager assignment style — was broken differently (squares stranded, lines fine)
p <- gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3)
p <- gf_square_resid(p, m_cat_complex, color = "blue")
p <- gf_resid(p, m_cat_complex, color = "red")
p

In [ ]:
# Test 11: native |> pipe — same laziness as %>%, should look like Test 5
gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3) |>
  gf_square_resid(m_cat_complex, color = "blue") |>
  gf_resid(m_cat_complex, color = "red")

## Re-rendering the same plot object

The old trick only survived one render — displaying the plot a second time (or `ggsave()`
after display) re-rolled the dots out from under the squares.

In [ ]:
# Test 12: same plot object rendered twice — the dots must be in identical
# positions in both plots, still under the squares
p <- gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3) %>%
  gf_square_resid(m_cat_complex, color = "blue")
p
p

## Non-jittered plots are untouched

In [ ]:
# Test 13: gf_point base — no jitter layer to freeze, behavior unchanged
gf_point(later_anxiety ~ condition, data = er_small, alpha = 0.4) %>%
  gf_model(m_cat_empty) %>%
  gf_model(m_cat_complex) %>%
  gf_square_resid(m_cat_complex, color = "red") %>%
  gf_square_reduce(m_cat_complex, color = "forestgreen")

In [ ]:
# Test 14: continuous x with jitter — full decomposition
gf_jitter(Thumb ~ Height, data = d, alpha = 0.4) %>%
  gf_model(m_empty) %>%
  gf_model(m_complex) %>%
  gf_square_resid(m_empty,    fill = "blue",  color = "blue",  alpha = 0.15) %>%
  gf_square_resid(m_complex,  fill = "red",   color = "red",   alpha = 0.15) %>%
  gf_square_reduce(m_complex, fill = "green", color = "green", alpha = 0.15)

## Seeds and the RNG stream

In [ ]:
# Test 15: user-supplied jitter seed is respected — re-run this cell several times;
# the dots should be in the SAME positions every run (seed = 5), overlays attached
gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1, alpha = 0.3, seed = 5) %>%
  gf_square_resid(m_cat_complex, color = "blue") %>%
  gf_resid(m_cat_complex, color = "red")

In [ ]:
# Test 16: numerical sanity — the freeze lives on the layer, not the global RNG
p <- gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1) %>%
  gf_resid(m_cat_complex, color = "red")
cat("jitter layer seed:", p$layers[[1]]$position$seed, "(finite -> frozen)\n")

# .freeze_jitter() never calls set.seed(), so two separate chains draw different
# jitters (the old code left the RNG parked at set.seed() of a number in 1:100)
x1 <- ggplot_build(gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1) %>%
                     gf_resid(m_cat_complex))$data[[1]]$x
x2 <- ggplot_build(gf_jitter(later_anxiety ~ condition, data = er_small, width = 0.1) %>%
                     gf_resid(m_cat_complex))$data[[1]]$x
cat("two independent chains draw different jitters:", !isTRUE(all.equal(x1, x2)), "\n")